# 🔮 Prévision des ventes avec Prophet
## Projet P9 — Librairie Lapage

---

Ce notebook présente **Prophet**, la bibliothèque de prévision de séries temporelles développée par Meta (Facebook).

### Objectifs

1. Comprendre le fonctionnement de Prophet
2. Préparer les données au format attendu
3. Entraîner un modèle de prévision
4. Visualiser les résultats
5. Évaluer les performances

### Pourquoi Prophet ?

| Avantage | Description |
|----------|-------------|
| **Simple** | Peu de paramètres à configurer |
| **Robuste** | Gère bien les données manquantes et les outliers |
| **Interprétable** | Décomposition tendance + saisonnalité |
| **Flexible** | Ajout de jours fériés, événements spéciaux |

---
## 1. Installation et imports

In [ ]:
# Installation de Prophet (décommenter si nécessaire)
# !pip install prophet plotly

In [2]:
# Imports
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Prophet
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics

# Visualisation
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Métriques
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Désactiver les warnings Prophet
import logging
logging.getLogger('prophet').setLevel(logging.WARNING)

print("✅ Imports réussis")

✅ Imports réussis


---
## 2. Chargement des données

In [3]:
# Charger les transactions
df = pd.read_csv('../data/processed/transactions_enrichies.csv')
df['date'] = pd.to_datetime(df['date'])

print(f"📊 {len(df):,} transactions chargées")
print(f"📅 Période : {df['date'].min().date()} → {df['date'].max().date()}")
df.head()

📊 687,534 transactions chargées
📅 Période : 2021-03-01 → 2023-02-28


,id_prod,date,session_id,client_id,price,categ,sex,birth,segment_client,age_client
0,0_1259,2021-03-01 00:01:07.843138,s_1,c_329,11.99,0,f,1967,B2C,59
1,0_1390,2021-03-01 00:02:26.047414,s_2,c_664,19.37,0,m,1960,B2C,66
2,0_1352,2021-03-01 00:02:38.311413,s_3,c_580,4.50,0,m,1988,B2C,38
3,0_1458,2021-03-01 00:04:54.559692,s_4,c_7912,6.55,0,f,1989,B2C,37
4,0_1358,2021-03-01 00:05:18.801198,s_5,c_2033,16.49,0,f,1956,B2C,70


---
## 3. Préparation des données pour Prophet

Prophet attend un DataFrame avec **exactement 2 colonnes** :
- `ds` : la date (datetime)
- `y` : la valeur à prédire (numérique)

Nous allons agréger les transactions par jour pour obtenir le **CA journalier**.

In [4]:
# Agrégation par jour : CA journalier
df_daily = df.groupby(pd.Grouper(key='date', freq='D')).agg(
    ca=('price', 'sum'),
    nb_transactions=('price', 'count')
).reset_index()

print(f"📅 {len(df_daily)} jours de données")
df_daily.head()

📅 730 jours de données


,date,ca,nb_transactions
0,2021-03-01,16565.22,962
1,2021-03-02,15486.45,939
2,2021-03-03,15198.69,911
3,2021-03-04,15196.07,903
4,2021-03-05,17471.37,943


In [5]:
# Format Prophet : renommer les colonnes
df_prophet = df_daily[['date', 'ca']].copy()
df_prophet.columns = ['ds', 'y']

print("✅ Format Prophet :")
print(f"   - ds : {df_prophet['ds'].dtype}")
print(f"   - y  : {df_prophet['y'].dtype}")
df_prophet.head()

✅ Format Prophet :
   - ds : datetime64[ns]
   - y  : float64


,ds,y
0,2021-03-01,16565.22
1,2021-03-02,15486.45
2,2021-03-03,15198.69
3,2021-03-04,15196.07
4,2021-03-05,17471.37


In [6]:
# Visualisation du CA journalier
fig = px.line(
    df_prophet, x='ds', y='y',
    title='📈 Chiffre d\'affaires journalier — Lapage',
    labels={'ds': 'Date', 'y': 'CA (€)'}
)
fig.update_layout(template='plotly_white')
fig.show()

---
## 4. Entraînement du modèle Prophet

### Fonctionnement de Prophet

Prophet décompose la série en 3 composantes :

$$y(t) = g(t) + s(t) + h(t) + \epsilon_t$$

| Composante | Description |
|------------|-------------|
| $g(t)$ | **Tendance** — évolution globale |
| $s(t)$ | **Saisonnalité** — patterns récurrents (hebdo, annuel) |
| $h(t)$ | **Jours fériés** — effets ponctuels |
| $\epsilon_t$ | **Bruit** — variations aléatoires |

In [7]:
# Création du modèle Prophet
model = Prophet(
    yearly_seasonality=True,   # Saisonnalité annuelle
    weekly_seasonality=True,   # Saisonnalité hebdomadaire
    daily_seasonality=False,   # Pas de saisonnalité journalière (données agrégées par jour)
    seasonality_mode='multiplicative'  # Saisonnalité proportionnelle au niveau
)

print("🔧 Modèle Prophet configuré")

🔧 Modèle Prophet configuré


In [8]:
# FIX : long windows path error
import os
import tempfile
# Forcer un dossier temp avec chemin court
os.environ['TMP'] = 'C:\\Temp'
os.environ['TEMP'] = 'C:\\Temp'
tempfile.tempdir = 'C:\\Temp'

# Créer le dossier si nécessaire
os.makedirs('C:\\Temp', exist_ok=True)

In [9]:
# Entraînement
model.fit(df_prophet)
print("✅ Modèle entraîné !")

16:36:13 - cmdstanpy - INFO - Chain [1] start processing
16:36:13 - cmdstanpy - INFO - Chain [1] done processing
16:36:13 - cmdstanpy - ERROR - Chain [1] error: code '3221225785' 
Optimization terminated abnormally. Falling back to Newton.
16:36:13 - cmdstanpy - INFO - Chain [1] start processing
16:36:13 - cmdstanpy - INFO - Chain [1] done processing
16:36:13 - cmdstanpy - ERROR - Chain [1] error: code '3221225785' 


RuntimeError: Error during optimization! Command 'C:\Users\RémiJulien\OneDrive\Documents\DcidConsulting\2.Prestation\3.Formation\3.Production\2025_GRETA DEV IA_P5\OCR_Projet_9\venv\Lib\site-packages\prophet\stan_model\prophet_model.bin random seed=29544 data file=C:\Temp\tmp6b1j6dh_\3w0vyqdb.json init=C:\Temp\tmp6b1j6dh_\cyaploi0.json output file=C:\Temp\tmp6b1j6dh_\prophet_modelrpc8ogvi\prophet_model-20260331163613.csv method=optimize algorithm=newton iter=10000' failed: 

---
## 5. Prévisions

Pour faire des prévisions, on crée un DataFrame des dates futures avec `make_future_dataframe()`.

In [10]:
# Créer un DataFrame avec les dates futures (30 jours)
future = model.make_future_dataframe(periods=30, freq='D')

print(f"📅 Dates générées : {len(future)}")
print(f"   Dernière date historique : {df_prophet['ds'].max().date()}")
print(f"   Dernière date prévision  : {future['ds'].max().date()}")
future.tail()

📅 Dates générées : 760
   Dernière date historique : 2023-02-28
   Dernière date prévision  : 2023-03-30


,ds
755,2023-03-26
756,2023-03-27
757,2023-03-28
758,2023-03-29
759,2023-03-30


In [11]:
# Générer les prévisions
forecast = model.predict(future)

# Colonnes principales
print("📊 Colonnes de sortie Prophet :")
print("   - ds       : date")
print("   - yhat     : prévision")
print("   - yhat_lower / yhat_upper : intervalle de confiance")
print("   - trend    : tendance")
print("   - weekly   : composante hebdomadaire")
print("   - yearly   : composante annuelle")

forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(10)

KeyError: 'k'

---
## 6. Visualisation des prévisions

### 6.1 Visualisation native Prophet

In [ ]:
# Graphique natif Prophet (matplotlib)
fig_prophet = model.plot(forecast)
fig_prophet.suptitle('Prévisions Prophet — CA Lapage', fontsize=14)

In [ ]:
# Décomposition des composantes
fig_components = model.plot_components(forecast)

### 6.2 Visualisation interactive avec Plotly

In [ ]:
def plot_forecast_plotly(df_actual, forecast, title="Prévisions"):
    """
    Visualisation interactive des prévisions avec Plotly.
    
    Parameters
    ----------
    df_actual : DataFrame avec colonnes 'ds' et 'y' (données réelles)
    forecast : DataFrame de sortie Prophet
    title : Titre du graphique
    """
    fig = go.Figure()
    
    # Intervalle de confiance
    fig.add_trace(go.Scatter(
        x=pd.concat([forecast['ds'], forecast['ds'][::-1]]),
        y=pd.concat([forecast['yhat_upper'], forecast['yhat_lower'][::-1]]),
        fill='toself',
        fillcolor='rgba(0, 100, 255, 0.15)',
        line=dict(color='rgba(255,255,255,0)'),
        name='Intervalle de confiance',
        showlegend=True
    ))
    
    # Prévisions
    fig.add_trace(go.Scatter(
        x=forecast['ds'],
        y=forecast['yhat'],
        mode='lines',
        name='Prévision',
        line=dict(color='#0066FF', width=2)
    ))
    
    # Données réelles
    fig.add_trace(go.Scatter(
        x=df_actual['ds'],
        y=df_actual['y'],
        mode='markers',
        name='Données réelles',
        marker=dict(color='#333333', size=4)
    ))
    
    # Ligne verticale : fin des données historiques
    last_date = df_actual['ds'].max()
    fig.add_vline(
        x=last_date, 
        line_dash="dash", 
        line_color="red",
        annotation_text="Début prévisions"
    )
    
    fig.update_layout(
        title=title,
        xaxis_title="Date",
        yaxis_title="CA (€)",
        template='plotly_white',
        hovermode='x unified',
        legend=dict(orientation='h', yanchor='bottom', y=1.02)
    )
    
    return fig

# Affichage
fig = plot_forecast_plotly(df_prophet, forecast, "🔮 Prévisions CA — Lapage (Prophet)")
fig.show()

In [ ]:
# Zoom sur les prévisions uniquement
last_date = df_prophet['ds'].max()
forecast_only = forecast[forecast['ds'] > last_date].copy()

fig = go.Figure()

# Intervalle de confiance
fig.add_trace(go.Scatter(
    x=pd.concat([forecast_only['ds'], forecast_only['ds'][::-1]]),
    y=pd.concat([forecast_only['yhat_upper'], forecast_only['yhat_lower'][::-1]]),
    fill='toself',
    fillcolor='rgba(0, 100, 255, 0.2)',
    line=dict(color='rgba(255,255,255,0)'),
    name='Intervalle de confiance'
))

# Prévision
fig.add_trace(go.Scatter(
    x=forecast_only['ds'],
    y=forecast_only['yhat'],
    mode='lines+markers',
    name='Prévision',
    line=dict(color='#0066FF', width=2)
))

fig.update_layout(
    title='📅 Prévisions des 30 prochains jours',
    xaxis_title='Date',
    yaxis_title='CA prévu (€)',
    template='plotly_white'
)

fig.show()

### 6.3 Analyse des composantes

In [ ]:
# Composantes en Plotly
fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=['Tendance', 'Saisonnalité hebdomadaire', 'Saisonnalité annuelle'],
    vertical_spacing=0.12
)

# Tendance
fig.add_trace(
    go.Scatter(x=forecast['ds'], y=forecast['trend'], mode='lines', name='Tendance', line=dict(color='#FF6B6B')),
    row=1, col=1
)

# Saisonnalité hebdomadaire (prendre une semaine type)
weekly = forecast[['ds', 'weekly']].copy()
weekly['dow'] = weekly['ds'].dt.dayofweek
weekly_avg = weekly.groupby('dow')['weekly'].mean().reset_index()
days = ['Lundi', 'Mardi', 'Mercredi', 'Jeudi', 'Vendredi', 'Samedi', 'Dimanche']
weekly_avg['jour'] = weekly_avg['dow'].map(lambda x: days[x])

fig.add_trace(
    go.Bar(x=weekly_avg['jour'], y=weekly_avg['weekly'], name='Hebdo', marker_color='#4ECDC4'),
    row=2, col=1
)

# Saisonnalité annuelle
fig.add_trace(
    go.Scatter(x=forecast['ds'], y=forecast['yearly'], mode='lines', name='Annuelle', line=dict(color='#45B7D1')),
    row=3, col=1
)

fig.update_layout(
    height=700,
    title_text='📊 Décomposition des composantes Prophet',
    showlegend=False,
    template='plotly_white'
)

fig.show()

---
## 7. Évaluation du modèle

### 7.1 Métriques sur l'ensemble d'entraînement

In [ ]:
# Fusionner prévisions et données réelles
df_eval = df_prophet.merge(forecast[['ds', 'yhat']], on='ds', how='left')

# Calcul des métriques
y_true = df_eval['y']
y_pred = df_eval['yhat']

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

print("📏 Métriques d'évaluation (sur données d'entraînement)")
print("="*50)
print(f"MAE  (Mean Absolute Error)      : {mae:,.2f} €")
print(f"RMSE (Root Mean Square Error)   : {rmse:,.2f} €")
print(f"MAPE (Mean Abs Percentage Error): {mape:.2f} %")
print("="*50)
print(f"\n💡 En moyenne, l'erreur de prévision est de {mae:,.0f}€ par jour")

### 7.2 Validation croisée temporelle

La validation croisée temporelle est plus robuste : on entraîne sur le passé et on prédit le futur.

```
┌─────────────────────────────┬──────────┐
│       Training              │   Test   │  Fold 1
├─────────────────────────────────┬──────┤
│           Training              │ Test │  Fold 2
├─────────────────────────────────────┬──┤
│               Training              │T │  Fold 3
└─────────────────────────────────────┴──┘
```

In [ ]:
# Validation croisée Prophet
# - initial : période d'entraînement initiale
# - period  : fréquence des cutoffs
# - horizon : horizon de prévision à évaluer

df_cv = cross_validation(
    model,
    initial='180 days',  # 6 mois d'entraînement initial
    period='30 days',    # Nouveau cutoff tous les 30 jours
    horizon='30 days'    # Prévisions sur 30 jours
)

print(f"✅ Validation croisée : {len(df_cv)} prévisions évaluées")
df_cv.head(10)

In [ ]:
# Métriques de performance
df_perf = performance_metrics(df_cv)

print("📊 Métriques par horizon de prévision :")
df_perf[['horizon', 'mae', 'rmse', 'mape']].head(10)

In [ ]:
# Visualisation des métriques par horizon
fig = px.line(
    df_perf, 
    x='horizon', 
    y=['mae', 'rmse'],
    title='📏 Évolution des erreurs selon l\'horizon de prévision',
    labels={'value': 'Erreur (€)', 'horizon': 'Horizon', 'variable': 'Métrique'}
)
fig.update_layout(template='plotly_white')
fig.show()

In [ ]:
# Résumé des métriques moyennes
print("\n" + "="*50)
print("📏 MÉTRIQUES MOYENNES (Validation croisée)")
print("="*50)
print(f"MAE  moyen : {df_perf['mae'].mean():,.2f} €")
print(f"RMSE moyen : {df_perf['rmse'].mean():,.2f} €")
print(f"MAPE moyen : {df_perf['mape'].mean()*100:.2f} %")
print("="*50)

---
## 8. Sauvegarde du modèle

Pour réutiliser le modèle dans l'API, on le sérialise avec `pickle` ou `joblib`.

In [ ]:
import pickle
from pathlib import Path

# Créer le dossier models si nécessaire
models_dir = Path('../models/saved')
models_dir.mkdir(parents=True, exist_ok=True)

# Sauvegarder le modèle
model_path = models_dir / 'prophet_model.pkl'

with open(model_path, 'wb') as f:
    pickle.dump(model, f)

print(f"✅ Modèle sauvegardé : {model_path}")
print(f"   Taille : {model_path.stat().st_size / 1024:.1f} Ko")

In [ ]:
# Vérification : recharger et tester
with open(model_path, 'rb') as f:
    model_loaded = pickle.load(f)

# Test de prévision
future_test = model_loaded.make_future_dataframe(periods=7)
forecast_test = model_loaded.predict(future_test)

print("✅ Modèle rechargé avec succès !")
print("\n📅 Prévisions des 7 prochains jours :")
forecast_test[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(7)

---
## 9. Fonction utilitaire pour l'API

Voici une fonction prête à être intégrée dans l'API FastAPI :

In [ ]:
def predict_ca(model, horizon_days: int = 30) -> dict:
    """
    Génère des prévisions de CA pour les N prochains jours.
    
    Parameters
    ----------
    model : Prophet
        Modèle Prophet entraîné
    horizon_days : int
        Nombre de jours à prédire
        
    Returns
    -------
    dict : Prévisions au format JSON-ready
    """
    # Générer les dates futures
    future = model.make_future_dataframe(periods=horizon_days)
    forecast = model.predict(future)
    
    # Filtrer uniquement les prévisions futures
    last_date = model.history['ds'].max()
    forecast_future = forecast[forecast['ds'] > last_date].copy()
    
    # Formater pour l'API
    predictions = []
    for _, row in forecast_future.iterrows():
        predictions.append({
            "date": row['ds'].strftime('%Y-%m-%d'),
            "ca_predicted": round(row['yhat'], 2),
            "lower": round(row['yhat_lower'], 2),
            "upper": round(row['yhat_upper'], 2)
        })
    
    return {
        "horizon_days": horizon_days,
        "predictions": predictions,
        "model_info": {
            "type": "Prophet",
            "last_training_date": last_date.strftime('%Y-%m-%d')
        }
    }

# Test
result = predict_ca(model_loaded, horizon_days=7)
print("📦 Format de sortie API :")
result

---
## 📝 Résumé

### Ce que nous avons fait

| Étape | Description |
|-------|-------------|
| **Préparation** | Agrégation CA journalier, format `ds`/`y` |
| **Entraînement** | Modèle Prophet avec saisonnalité hebdo + annuelle |
| **Prévision** | 30 jours avec intervalle de confiance |
| **Visualisation** | Plotly interactif + décomposition |
| **Évaluation** | MAE, RMSE, MAPE + validation croisée |
| **Sauvegarde** | `prophet_model.pkl` |

### Prochaines étapes (Phase 5)

1. **Intégrer à l'API** : Endpoint `POST /api/predict`
2. **Consommer depuis le dashboard** : Appel API + visualisation
3. **Améliorer le modèle** : Ajouter jours fériés, ajuster hyperparamètres

---

✅ **Notebook terminé** — Prêt pour l'intégration API !